# HippoCortex on Vim-small, Split-CIFAR100 (10x10): the baselines' protocol

PR #23 moved HippoCortex onto Vim-small with the same protocol as the Inf-SSM and Mamba-CL
baselines (natural class order, the authors' task-0 checkpoint, shared 100-way head, 10
tasks x 10 classes, seed 0), so HippoCortex can sit in the same comparison table
(`experiments/stage1_split_cifar100/eval_baseline.py --table`).

**Entry point:** `experiments/stage1_split_cifar100/run.py` (OmegaConf config +
command-line overrides), not `baselines/*/train_eval.py` like the other notebooks in this
folder. Each task's W&B row holds `AA`/`AIA`/`FM` plus the per-task accuracy matrix
(`acc/task_<j>` for every task seen so far) — the same shape as the Inf-SSM baselines'
`test/class_<i>_acc` columns, so `eval_baseline.py --wandb_run .../hippocortex-stage1/<id>
--method hippocortex` can fill the HippoCortex row, BWT included, with no local results
file.

**Settings:** this run trains about 9.7M parameters on Vim-small (vs Inf-SSM's 26M and
Mamba-CL's ~10M), 18 epochs/task instead of 40, batch size 64 instead of 128 — a single T4
is enough for this notebook. `run.py` does not yet support multi-GPU (`DataParallel`) or
cross-session checkpoint resume the way the Inf-SSM/Mamba-CL notebooks do (they mirror
checkpoints to a private HF repo so a 12h-capped Kaggle session can continue a multi-day
run). If a full 10-task run does not fit in one session, that resume support is the next
piece of infrastructure to add — out of scope for this notebook.

Attach as a Kaggle Secret (Add-ons > Secrets):
- `WANDB_API_KEY` (optional). Without it, training still runs locally; it just is not
  logged to W&B, so `eval_baseline.py` has nothing to read for this run. Attach it before
  the run you want in the comparison table.

**First run on a new setup:** run the smoke-test cell first (`RUN_SMOKE_TEST = True`):
2 tasks, 0 warmup epochs, 1 joint epoch, W&B disabled. It should finish in a few minutes
and print `Done. AA=... AIA=... FM=...` with no errors before you run the full training
cell.


In [ ]:
# ---- run parameters ----
REPO    = "https://github.com/HasinthakaPiyumal/hippo-cortex.git"
BRANCH  = "main"
RUN_SMOKE_TEST = True      # 2-task sanity check (no W&B) before the full run; set False once it has passed
WANDB_PROJECT  = "hippocortex-stage1"
WANDB_ENTITY   = "hasinthakapiyumal-uok"
SEED = 0                   # paper / baselines: 0 (the only seed with a released task-0 checkpoint)


In [ ]:
!nvidia-smi
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("hippo-cortex"):
    # Fall back to main once the branch is merged (GitHub deletes merged branches here).
    if subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO]).returncode != 0:
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
ROOT = "/kaggle/working/hippo-cortex"
VENV = "/kaggle/working/py311_env"
PY, PIP = f"{VENV}/bin/python", f"{VENV}/bin/pip"
!cd {ROOT} && git log --oneline -1


In [ ]:
# Python 3.11 venv (the Kaggle image's Python is too new for torch 2.3.1 / mamba-ssm 1.2.2).
# Install order matters: mamba-ssm and causal-conv1d import torch at build time
# (see requirements.txt), so torch must already be installed before them.
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null
!apt-get update -y -qq > /dev/null
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev python3.11-distutils > /dev/null
!python3.11 -m venv {VENV}
!{PY} -m ensurepip
!{PIP} install -q --upgrade pip
!{PIP} install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -q ninja packaging wheel
!{PIP} install -q --no-build-isolation causal-conv1d==1.4.0
!{PIP} install -q --no-build-isolation mamba-ssm==1.2.2
!cd {ROOT} && {PIP} install -q -r requirements.txt
!cd {ROOT} && {PIP} install -q -e .
!{PIP} install -q -U wandb
# check=True stops Run All here if the kernels are missing (a failing "!" command would not).
subprocess.run([PY, "-c",
                "import torch, mamba_ssm, causal_conv1d; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "torch.cuda.device_count(), 'GPU(s) | mamba_ssm', mamba_ssm.__version__)"],
               check=True)


In [ ]:
# Tokens come from Kaggle Secrets; they are never written into this notebook.
env = dict(os.environ)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
try:
    env["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    use_wandb = True
    print("W&B: online (key from Kaggle Secret)")
except Exception as e:
    use_wandb = False
    print("W&B: secret WANDB_API_KEY not attached — this run will not be logged:", type(e).__name__)


In [ ]:
# Smoke test: 2 tasks, 0 warmup epochs, 1 joint epoch, no W&B. Confirms the environment
# and data download work before committing a session to the full 10-task run.
if RUN_SMOKE_TEST:
    smoke_cmd = [PY, "-u", "experiments/stage1_split_cifar100/run.py",
                 f"seed={SEED}", "dataset.n_tasks=2",
                 "training.epochs_warmup=0", "training.epochs_joint=1",
                 "logging.use_wandb=false"]
    print(" ".join(smoke_cmd))
    subprocess.run(smoke_cmd, cwd=ROOT, check=True)
else:
    print("RUN_SMOKE_TEST is False — skipped.")


In [ ]:
# Full run: all 10 tasks, the paper's epoch counts (config.yaml), logged to
# WANDB_PROJECT under WANDB_ENTITY when the WANDB_API_KEY secret is attached.
import time
LOG = f"/kaggle/working/hippocortex_cifar100_seed_{SEED}_{time.strftime('%Y%m%d-%H%M%S')}.log"
full_cmd = [PY, "-u", "experiments/stage1_split_cifar100/run.py",
            f"seed={SEED}",
            f"logging.use_wandb={'true' if use_wandb else 'false'}",
            f"logging.wandb_project={WANDB_PROJECT}",
            f"logging.wandb_entity={WANDB_ENTITY}"]
print(" ".join(full_cmd))
t0 = time.time()
with open(LOG, "w") as f:
    p = subprocess.Popen(full_cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        f.write(line); f.flush()
        print(line, end="", flush=True)   # full log: per-task phases, eval accuracy, AA/AIA/FM
    p.wait()
print("exit code", p.returncode, f"in {(time.time() - t0) / 3600:.2f} h; full log: {LOG}")


In [ ]:
# Show results so far, next to the known baseline numbers (same protocol, seed 0).
import json
res_path = f"{ROOT}/results/stage1_split_cifar100/summary.json"
if os.path.exists(res_path):
    r = json.load(open(res_path))
    print(f"HippoCortex: AA {r['AA']:.2%} | AIA {r['AIA']:.2%} | FM {r['FM']:.2%}")
    print("Inf-SSM:                 AA 28.42% | AIA 55.35% | FM 24.83%")
    print("Mamba-CL (shared head):   AA 51.52% | AIA 59.43% | FM 48.62%")
else:
    print("no summary.json yet")


In [ ]:
!tail -n 40 {LOG}
